# IK Loss Testing Notebook

**Location:** `/home/xray/Nicki/BOCSER/ik_loss_test.ipynb`  
**Run:** `jupyter notebook` from `/home/xray/Nicki/BOCSER/`

Tests:
1. IK Loss at reference geometry (should be ~0)
2. IK Loss landscape over dihedral grid
3. Correlation with real ring openings
4. A/B comparison: IK Loss vs Closure Distance Loss

## Cell 0 — Imports

In [2]:
import sys, os, tempfile, warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['AUTOGRAPH_VERBOSITY'] = '0'

sys.path.insert(0, os.path.join(os.getcwd(), 'bocser'))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd
from IPython.display import display

tf.autograph.set_verbosity(0)
from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()
tf.config.run_functions_eagerly(True)

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

import config_manager
from default_vals import ConfSearchConfig
from ik_loss import IKLoss, CyclicCollection
from coef_calc import CoefCalculator
from calc import dihedral_angle

cfg = ConfSearchConfig(
    mol_file_name='tests/NonBR/NonBR.mol',
    orca_method='XTB2',
    acquisition_function='ik',
    ts=True,
)
config_manager.set_config(cfg)
print('OK — imports done')
print(f'TF: {tf.__version__}')

OK — imports done
TF: 2.16.2


## Cell 1 — Load molecule, build IKLoss

In [3]:
MOL_PATH = 'tests/NonBR/NonBR.mol'

mol_raw = Chem.MolFromMolFile(MOL_PATH, removeHs=False)
mol     = Chem.RemoveHs(mol_raw)

_tmp_scans_dir = tempfile.mkdtemp(prefix='ik_test_scans_')

coef_calc = CoefCalculator(
    mol=mol,
    config=cfg,
    dir_for_inps=_tmp_scans_dir,
    db_connector=None,
)

try:
    all_dihedrals, all_ring_traversals, ik_loss_idxs = coef_calc.get_ring_dihedrals(mol)
    print(f'Rings found: {len(all_ring_traversals)}')
    for i, ring in enumerate(all_ring_traversals):
        print(f'  Ring {i}: {len(ring)} atoms — {ring}')
    print(f'IK dihedral indices: {ik_loss_idxs}')
except Exception as e:
    print(f'Error: {e}')
    raise

ik_loss_fn = IKLoss.from_rdkit(mol, all_ring_traversals)
print(f'\nIKLoss built — {len(ik_loss_fn.D_matrices)} rings')

# Collect reference dihedrals from .mol file conformer
# Shape per ring: [n_dihedrals_in_ring]  (flat list of angles)
ref_angles_per_ring = []
ref_data = []
for i, ring in enumerate(all_ring_traversals):
    ring_c = CyclicCollection(ring)
    angles = []
    for k in range(len(ring_c)):
        atoms = tuple(ring_c[k + s] for s in (-1, 0, 1, 2))
        positions = [mol_raw.GetConformer().GetAtomPosition(a) for a in atoms]
        angle = dihedral_angle(*positions)
        angles.append(angle)
        ref_data.append({'Ring': i, 'Atoms': str(atoms), 'Ref dihedral (rad)': round(angle, 4)})
    ref_angles_per_ring.append(angles)

print('\nReference dihedrals from .mol file:')
display(pd.DataFrame(ref_data))

def make_tensors(angles_per_ring):
    """Convert list-of-lists of angles to list of tf.Tensor with shape [1, n_dihedrals].
    build_T_matrices expects [batch, n_dihedrals] — batch=1 for single-point evaluation.
    """
    return [
        tf.constant([ring_angles], dtype=tf.float64)   # shape [1, n_dihedrals]
        for ring_angles in angles_per_ring
    ]

print('\nmake_tensors helper defined — tensor shapes:')
for i, t in enumerate(make_tensors(ref_angles_per_ring)):
    print(f'  Ring {i}: {t.shape}  (expected [1, {len(ref_angles_per_ring[i])}])')

Rings found: 3
  Ring 0: 15 atoms — [14, 11, 2, 3, 4, 0, 1, 5, 6, 7, 8, 9, 10, 12, 13]
  Ring 1: 6 atoms — [17, 16, 15, 12, 13, 18]
  Ring 2: 5 atoms — [19, 17, 18, 21, 20]
IK dihedral indices: [[-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1], [-1, -1, -1, -1, -1, -1], [-1, -1, -1, -1, -1]]

IKLoss built — 3 rings

Reference dihedrals from .mol file:


,Ring,Atoms,Ref dihedral (rad)
0,0,"(13, 14, 11, 2)",3.2443
1,0,"(14, 11, 2, 3)",2.9033
2,0,"(11, 2, 3, 4)",2.6972
3,0,"(2, 3, 4, 0)",5.3713
4,0,"(3, 4, 0, 1)",5.5178
5,0,"(4, 0, 1, 5)",3.2395
6,0,"(0, 1, 5, 6)",3.2762
7,0,"(1, 5, 6, 7)",1.0416
8,0,"(5, 6, 7, 8)",4.5538
9,0,"(6, 7, 8, 9)",2.9272



make_tensors helper defined — tensor shapes:
  Ring 0: (1, 15)  (expected [1, 15])
  Ring 1: (1, 6)  (expected [1, 6])
  Ring 2: (1, 5)  (expected [1, 5])


## Cell 2 — Test 1: IK Loss at reference geometry (should be ~0)

In [ ]:
ref_tensors = make_tensors(ref_angles_per_ring)
loss_at_ref = float(ik_loss_fn(ref_tensors).numpy())

print('=' * 50)
print('TEST 1: IK Loss at reference geometry')
print('=' * 50)
print(f'Expected : ~0.0 (ring is closed in .mol file)')
print(f'Got      : {loss_at_ref:.6f}')
print()
if loss_at_ref < 0.1:
    print('PASSED — IKLoss correctly returns ~0 for closed ring')
elif loss_at_ref < 1.0:
    print('MARGINAL — small non-zero loss; likely numerical precision or 2D-tagged mol')
else:
    print('FAILED — IKLoss is large at reference geometry')
    print('  Possible causes: mol tagged as 2D, bad coordinates in .mol file')

NameError: name 'ik_loss' is not defined

## Cell 3 — Test 2: IK Loss landscape over dihedral grid (ring 0, dihedrals 0 and 1)

In [ ]:
RING_IDX = 0
N_GRID   = 30   # 30x30 = 900 evaluations — fast enough

phi_range   = np.linspace(0, 2 * np.pi, N_GRID)
grid_losses = np.zeros((N_GRID, N_GRID))

ref_ring_0 = ref_angles_per_ring[RING_IDX].copy()
print(f'Scanning dihedrals 0 and 1 of ring {RING_IDX} ({len(ref_ring_0)} dihedrals total)...')

for i, phi1 in enumerate(phi_range):
    for j, phi2 in enumerate(phi_range):
        varied = ref_ring_0.copy()
        varied[0] = phi1
        if len(varied) > 1:
            varied[1] = phi2

        # Build tensors: shape [1, n_dihedrals] per ring
        tensors = []
        for ridx, ref_angles in enumerate(ref_angles_per_ring):
            angles = varied if ridx == RING_IDX else ref_angles
            tensors.append(tf.constant([angles], dtype=tf.float64))

        grid_losses[i, j] = float(ik_loss_fn(tensors).numpy())

print('Done.')

ref_phi1 = ref_ring_0[0]
ref_phi2 = ref_ring_0[1] if len(ref_ring_0) > 1 else 0.0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

im1 = axes[0].contourf(phi_range, phi_range, grid_losses, levels=50, cmap='viridis')
axes[0].scatter([ref_phi2], [ref_phi1], color='red', s=100, zorder=5, label='Ref (.mol)')
axes[0].set_xlabel('Dihedral 1 (rad)'); axes[0].set_ylabel('Dihedral 0 (rad)')
axes[0].set_title(f'IK Loss — linear scale (ring {RING_IDX})')
axes[0].legend(); plt.colorbar(im1, ax=axes[0])

log_losses = np.log1p(grid_losses)
im2 = axes[1].contourf(phi_range, phi_range, log_losses, levels=50, cmap='viridis')
axes[1].scatter([ref_phi2], [ref_phi1], color='red', s=100, zorder=5, label='Ref (.mol)')
axes[1].set_xlabel('Dihedral 1 (rad)'); axes[1].set_ylabel('Dihedral 0 (rad)')
axes[1].set_title(f'IK Loss — log scale (ring {RING_IDX})')
axes[1].legend(); plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.savefig('ik_loss_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

display(pd.DataFrame([{
    'min': grid_losses.min(), 'max': grid_losses.max(),
    'mean': grid_losses.mean(), 'std': grid_losses.std(),
    'loss at ref': float(ik_loss_fn(make_tensors(ref_angles_per_ring)).numpy()),
}]).round(6))

## Cell 4 — Test 3: Correlation with real ring openings (_check_rings_intact)

In [ ]:
from rdkit.Chem import rdMolTransforms
from calc import _check_rings_intact

mol_with_h = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol_with_h, randomSeed=42)

def apply_dihedrals_to_mol(base_mol, angles_per_ring, all_ring_traversals):
    """Apply dihedral angles to a copy of mol and return xyz block."""
    mol_copy = Chem.RWMol(base_mol)
    conf = mol_copy.GetConformer()
    for ridx, ring in enumerate(all_ring_traversals):
        ring_c = CyclicCollection(ring)
        for k, angle in enumerate(angles_per_ring[ridx]):
            atoms = tuple(ring_c[k + s] for s in (-1, 0, 1, 2))
            try:
                rdMolTransforms.SetDihedralRad(conf, *atoms, float(angle))
            except Exception:
                pass
    return Chem.MolToXYZBlock(mol_copy)

N_SAMPLE = 200
rng = np.random.default_rng(42)
sample_results = []

for _ in range(N_SAMPLE):
    # Random dihedrals for all rings — shape: list of [n_dihedrals] per ring
    rand_angles = [rng.uniform(0, 2*np.pi, len(r)).tolist() for r in ref_angles_per_ring]
    tensors = [tf.constant([a], dtype=tf.float64) for a in rand_angles]

    ik_val = float(ik_loss_fn(tensors).numpy())

    try:
        xyz = apply_dihedrals_to_mol(mol_with_h, rand_angles, all_ring_traversals)
        ring_ok = _check_rings_intact(xyz, mol)
    except Exception:
        ring_ok = None

    sample_results.append({'ik_loss': ik_val, 'ring_intact': ring_ok})

df_corr = pd.DataFrame(sample_results).dropna()
intact = df_corr[df_corr['ring_intact'] == True]['ik_loss']
broken = df_corr[df_corr['ring_intact'] == False]['ik_loss']

summary = df_corr.groupby('ring_intact')['ik_loss'].agg(['mean','median','std','count'])
summary.index = summary.index.map({True: 'Ring intact', False: 'Ring broken'})
summary.columns = ['Mean IK Loss', 'Median', 'Std', 'Count']
print('TEST 3: IK Loss vs ring integrity')
print('=' * 50)
display(summary.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(intact, bins=30, alpha=0.7, label='Intact', color='steelblue')
axes[0].hist(broken, bins=30, alpha=0.7, label='Broken', color='tomato')
axes[0].set_xlabel('IK Loss'); axes[0].set_title('Distribution'); axes[0].legend()

axes[1].boxplot(
    [intact.values, broken.values], labels=['Intact', 'Broken'],
    patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.7)
)
axes[1].set_ylabel('IK Loss'); axes[1].set_title('Box plot')
plt.tight_layout()
plt.savefig('ik_loss_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

if len(intact) > 0 and len(broken) > 0:
    thr = (intact.mean() + broken.mean()) / 2
    tp = (broken > thr).sum(); fp = (intact > thr).sum()
    fn = (broken <= thr).sum(); tn = (intact <= thr).sum()
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    print(f'Threshold={thr:.3f}  Precision={prec:.3f}  Recall={rec:.3f}  F1={f1:.3f}')
    print('GOOD' if f1 > 0.7 else 'POOR — consider alternative loss')

## Cell 5 — Alternative: Closure Distance Loss

In [ ]:
try:
    from calc import _VDW_RADII, _VDW_RADII_DEFAULT
except ImportError:
    # Fallback if not yet in calc.py
    _VDW_RADII = {'H':1.20,'C':1.70,'N':1.55,'O':1.52,'F':1.47,
                  'P':1.80,'S':1.80,'Cl':1.75,'Br':1.85,'I':1.98}
    _VDW_RADII_DEFAULT = 1.70

class ClosureDistanceLoss:
    """Penalises deviation of ring-closing bond distances from their reference values.
    loss = sum_rings( (dist - r_ref)^2 * exp(max(0, dist - 0.7*vdw)) )
    The exponential weight grows sharply when the bond is clearly broken.
    """
    def __init__(self, mol, all_ring_traversals):
        self.mol_with_h = Chem.AddHs(mol)
        AllChem.EmbedMolecule(self.mol_with_h, randomSeed=42)
        self.closing_pairs  = []
        self.ref_distances  = []
        self.vdw_thresholds = []
        for ring in all_ring_traversals:
            ring_c = CyclicCollection(ring)
            a_idx, b_idx = ring_c[-1], ring_c[0]
            self.closing_pairs.append((a_idx, b_idx))
            pa = np.array(mol.GetConformer().GetAtomPosition(a_idx))
            pb = np.array(mol.GetConformer().GetAtomPosition(b_idx))
            self.ref_distances.append(float(np.linalg.norm(pa - pb)))
            sa = mol.GetAtomWithIdx(a_idx).GetSymbol()
            sb = mol.GetAtomWithIdx(b_idx).GetSymbol()
            vdw = _VDW_RADII.get(sa,_VDW_RADII_DEFAULT) + _VDW_RADII.get(sb,_VDW_RADII_DEFAULT) + 0.1
            self.vdw_thresholds.append(vdw)
            print(f'  Ring closing bond {a_idx}({sa})-{b_idx}({sb}): r_ref={self.ref_distances[-1]:.3f}A  vdw={vdw:.3f}A')

    def __call__(self, angles_per_ring, all_ring_traversals):
        """angles_per_ring: list of plain Python lists (not tensors)."""
        from rdkit.Chem import rdMolTransforms
        total = 0.0
        for ring_idx, (a_idx, b_idx) in enumerate(self.closing_pairs):
            mol_copy = Chem.RWMol(self.mol_with_h)
            conf = mol_copy.GetConformer()
            ring_c = CyclicCollection(all_ring_traversals[ring_idx])
            for k, angle in enumerate(angles_per_ring[ring_idx]):
                atoms = tuple(ring_c[k + s] for s in (-1, 0, 1, 2))
                try:
                    rdMolTransforms.SetDihedralRad(conf, *atoms, float(angle))
                except Exception:
                    pass
            pa = np.array(mol_copy.GetConformer().GetAtomPosition(a_idx))
            pb = np.array(mol_copy.GetConformer().GetAtomPosition(b_idx))
            dist = float(np.linalg.norm(pa - pb))
            dev  = dist - self.ref_distances[ring_idx]
            w    = np.exp(max(0.0, dist - self.vdw_thresholds[ring_idx] * 0.7))
            total += dev**2 * w
        return total

closure_loss = ClosureDistanceLoss(mol, all_ring_traversals)

## Cell 6 — A/B comparison: IK Loss vs Closure Distance Loss

In [ ]:
N_COMPARE = 300
rng2 = np.random.default_rng(123)
compare_results = []

for _ in range(N_COMPARE):
    rand_angles = [rng2.uniform(0, 2*np.pi, len(r)).tolist() for r in ref_angles_per_ring]

    # IK Loss — tensors shape [1, n_dihedrals]
    tensors = [tf.constant([a], dtype=tf.float64) for a in rand_angles]
    ik_val  = float(ik_loss_fn(tensors).numpy())

    # Closure Loss — plain Python lists
    cl_val  = closure_loss(rand_angles, all_ring_traversals)

    try:
        xyz      = apply_dihedrals_to_mol(mol_with_h, rand_angles, all_ring_traversals)
        ring_ok  = _check_rings_intact(xyz, mol)
    except Exception:
        ring_ok  = None

    compare_results.append({'IK Loss': ik_val, 'Closure Loss': cl_val, 'ring_intact': ring_ok})

df_ab = pd.DataFrame(compare_results).dropna()
intact_m = df_ab['ring_intact'] == True
broken_m = df_ab['ring_intact'] == False
print(f'Points: {len(df_ab)}  intact={intact_m.sum()}  broken={broken_m.sum()}')

rows = []
for col in ['IK Loss', 'Closure Loss']:
    iv = df_ab.loc[intact_m, col]; bv = df_ab.loc[broken_m, col]
    thr = (iv.mean() + bv.mean()) / 2
    tp = (bv>thr).sum(); fp = (iv>thr).sum(); fn = (bv<=thr).sum()
    prec = tp/(tp+fp) if (tp+fp)>0 else 0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0
    sep  = bv.mean()/iv.mean() if iv.mean()>0 else float('inf')
    rows.append({'Loss': col, 'Mean(intact)': round(iv.mean(),4),
                 'Mean(broken)': round(bv.mean(),4), 'Ratio': round(sep,2),
                 'Precision': round(prec,3), 'Recall': round(rec,3), 'F1': round(f1,3)})

df_metrics = pd.DataFrame(rows).set_index('Loss')
print('\nMetrics:')
display(df_metrics)

fig = plt.figure(figsize=(16, 9))
gs  = GridSpec(2, 3, figure=fig)

for ax_idx, col in enumerate(['IK Loss', 'Closure Loss']):
    ax = fig.add_subplot(gs[0, ax_idx])
    ax.hist(df_ab.loc[intact_m, col], bins=30, alpha=0.7, label='Intact', color='steelblue')
    ax.hist(df_ab.loc[broken_m, col], bins=30, alpha=0.7, label='Broken', color='tomato')
    ax.set_title(f'{col} distribution'); ax.set_xlabel(col); ax.legend(fontsize=8)

ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(df_ab.loc[intact_m,'IK Loss'], df_ab.loc[intact_m,'Closure Loss'],
            alpha=0.4, s=15, color='steelblue', label='Intact')
ax3.scatter(df_ab.loc[broken_m,'IK Loss'], df_ab.loc[broken_m,'Closure Loss'],
            alpha=0.4, s=15, color='tomato',    label='Broken')
ax3.set_xlabel('IK Loss'); ax3.set_ylabel('Closure Loss')
ax3.set_title('IK vs Closure Loss scatter'); ax3.legend(fontsize=8)

for ax_idx, col in enumerate(['IK Loss', 'Closure Loss']):
    ax = fig.add_subplot(gs[1, ax_idx])
    ax.boxplot([df_ab.loc[intact_m,col].values, df_ab.loc[broken_m,col].values],
               labels=['Intact','Broken'], patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.7))
    ax.set_title(f'{col} box plot')

ax6 = fig.add_subplot(gs[1, 2])
mets = ['Precision','Recall','F1']
x = np.arange(len(mets)); w = 0.35
ax6.bar(x-w/2, df_metrics.loc['IK Loss',     mets], w, label='IK Loss',      color='steelblue', alpha=0.8)
ax6.bar(x+w/2, df_metrics.loc['Closure Loss', mets], w, label='Closure Loss', color='tomato',    alpha=0.8)
ax6.set_xticks(x); ax6.set_xticklabels(mets); ax6.set_ylim(0,1.1)
ax6.axhline(0.7, color='gray', linestyle='--', alpha=0.5)
ax6.set_title('Quality metrics'); ax6.legend(fontsize=8)

plt.suptitle('A/B: IK Loss vs Closure Distance Loss', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ik_loss_ab_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

winner = df_metrics['F1'].idxmax()
print(f'\nWinner by F1: {winner}  (F1={df_metrics.loc[winner,"F1"]:.3f})')

## Cell 7 — Summary

In [ ]:
print('=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Molecule : {MOL_PATH}')
print(f'Rings    : {len(all_ring_traversals)}')
print()
print('IK dihedral coverage:')
for ri, idxs in enumerate(ik_loss_idxs):
    mapped = sum(1 for x in idxs if x != -1)
    total  = len(idxs)
    pct    = 100*mapped//total if total else 0
    print(f'  Ring {ri}: {mapped}/{total} ({pct}%)')
print()
print(f'IK Loss at reference geometry : {loss_at_ref:.6f}  (ideal = 0)')
print()
print('A/B metrics:')
display(df_metrics)
print()
winner = df_metrics['F1'].idxmax()
best_f1 = df_metrics.loc[winner,'F1']
if best_f1 >= 0.7:
    print(f'Recommended loss: {winner}  (F1={best_f1:.3f})')
else:
    print(f'Both losses show F1 < 0.7 — consider improving IK dihedral mapping coverage')
print()
print('Saved: ik_loss_landscape.png  ik_loss_correlation.png  ik_loss_ab_comparison.png')